In [40]:
from datetime import datetime
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np

def prt(df):
    print(df.to_string())

def filter_df(df: pd.DataFrame, **filters) -> pd.DataFrame:
    out = df

    for col, value in filters.items():
        if col not in df: 
            raise KeyError(f"Missing column: {col}")

        values = value if isinstance(value, (list, tuple, set)) else [value]
        valid = sorted(out[col].dropna().unique())

        out = out[out[col].isin(values)]

        if out.empty:
            print(
                f"{col} looking for {values}. \n"
                f"Valid options: {valid}"
            )
    prt(out)
    return out.copy()

In [41]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib
import re

import pandas as pd


MATRIX_PRESET = "cartesian"
USE_USPLAT_DEFAULT = False
DROPOUT_DEFAULT = "no_dropout"


FINAL_COLUMNS = [
    "status",
    "run_hash",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "config_path",
    "generated_config_path",
    "model_path",
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_requested_split",
    "eval_split_used",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_bytes",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
    "metrics_jsonl_path",
    "metrics_csv_path",
    "created_at",
]


DATASET_OVERRIDES = {
    "ablations_usplat_sort_dropout_off/trex": {
        "use_usplat": True,
    },
    "ablations_no_usplat_dropout_on/trex": {
        "dropout": "yes_dropout",
    },
}

INFER_USPLAT_KEYS = {
    "ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs",
}


DROP_RULES = [
    ("ablations_no_usplat_dropout_off/trex", "sorting", "sort_free"),
]


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def is_missing(value) -> bool:
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return isinstance(value, str) and value.strip() == ""

def value_or(record: dict, key: str, fallback):
    value = record.get(key, pd.NA)
    return fallback if is_missing(value) else value


def parse_variant_dirname(dirname: str) -> dict:
    """
    Supports:

    5-part:
        isotropic--rgb--sort--no_pruning--no_ess

    6-part:
        isotropic--rgb--sort--no_pruning--no_dropout--no_ess

    7-part:
        anisotropic--rgb--sort--interleaved_prune_densify--ess--dropout--no_usplat
    """
    parts = dirname.split("--")
    use_usplat = USE_USPLAT_DEFAULT

    if len(parts) == 5:
        isotropy, appearance, sorting, pruning, ess = parts
        dropout = DROPOUT_DEFAULT

    elif len(parts) == 6:
        isotropy, appearance, sorting, pruning, dropout, ess = parts

    elif len(parts) == 7:
        isotropy, appearance, sorting, pruning, ess, dropout, usplat = parts
        use_usplat = usplat != "no_usplat"

    else:
        raise ValueError(f"Unexpected variant folder name: {dirname}")

    variant_name = "__".join(
        [isotropy, appearance, sorting, pruning, dropout, ess]
    )

    return {
        "isotropy": isotropy,
        "appearance": appearance,
        "sorting": sorting,
        "pruning": pruning,
        "dropout": dropout,
        "ess": ess,
        "use_usplat": use_usplat,
        "variant_name": variant_name,
    }


def clean_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("string").str.strip()

    return df


def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def add_mb_columns(df: pd.DataFrame) -> pd.DataFrame:
    byte_cols = [
        col for col in df.columns
        if col.endswith("_bytes") or "bytes" in col.lower()
    ]

    for byte_col in byte_cols:
        mb_col = byte_col.replace("_bytes", "_mb")
        mb_values = pd.to_numeric(df[byte_col], errors="coerce") / (1024 ** 2)

        if mb_col in df.columns:
            df[mb_col] = df[mb_col].where(pd.notna(df[mb_col]), mb_values)
        else:
            df[mb_col] = mb_values

    return df


def build_record(
    row: pd.Series,
    *,
    variant_dir: Path,
    variant_index: int,
    parsed: dict,
    scene_name: str,
    config_path: str,
    created_at: str,
) -> dict:
    record = row.to_dict()

    defaults = {
        "run_hash": stable_hash(str(variant_dir.resolve())),
        "scene_name": scene_name,
        "variant_name": parsed["variant_name"],
        "matrix_preset": MATRIX_PRESET,
        "model_path": str(variant_dir.resolve()),
        "metrics_csv_path": str((variant_dir / "checkpoint_eval_metrics.csv").resolve()),
        "metrics_jsonl_path": str((variant_dir / "checkpoint_eval_metrics.jsonl").resolve()),
        "training_wall_clock_sec_at_checkpoint": pd.NA,
    }

    for key in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess", "use_usplat"]:
        defaults[key] = parsed[key]

    for key, fallback in defaults.items():
        record[key] = value_or(record, key, fallback)

    record["variant_index"] = variant_index
    record["config_path"] = config_path
    record["created_at"] = created_at

    # chkpnt_best.pth may have no number in filename, so use loaded eval iteration.
    record["checkpoint_name_iteration"] = value_or(
        record,
        "checkpoint_name_iteration",
        record.get("eval_checkpoint_iteration", pd.NA),
    )

    return record


def read_dataset_results(dataset_dir: Path, write_csv: bool = True) -> pd.DataFrame:
    dataset_dir = Path(dataset_dir)
    scene_name = dataset_dir.name

    config_path = f"configs/dnerf_ablation/{scene_name}.yaml"
    output_csv = dataset_dir.parent / f"{scene_name}_checkpoint_eval_metrics_all.csv"

    rows = []
    created_at = datetime.now().replace(microsecond=0).isoformat()

    variant_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    for variant_index, variant_dir in enumerate(variant_dirs):
        metrics_csv = variant_dir / "checkpoint_eval_metrics.csv"

        if not metrics_csv.exists():
            print(f"Skipping, no metrics file: {metrics_csv}")
            continue

        parsed = parse_variant_dirname(variant_dir.name)
        df = clean_string_columns(pd.read_csv(metrics_csv, skipinitialspace=True))

        for _, row in df.iterrows():
            rows.append(
                build_record(
                    row,
                    variant_dir=variant_dir,
                    variant_index=variant_index,
                    parsed=parsed,
                    scene_name=scene_name,
                    config_path=config_path,
                    created_at=created_at,
                )
            )

    combined = pd.DataFrame(rows)
    combined = ensure_columns(combined, FINAL_COLUMNS)
    combined = add_mb_columns(combined)
    combined = ensure_columns(combined, FINAL_COLUMNS)

    final_df = combined[FINAL_COLUMNS].copy()

    if write_csv:
        final_df.to_csv(output_csv, index=False)
        print(f"Wrote {len(final_df)} rows to {output_csv}")

    return final_df


def read_all_dataset_results(
    output_root: Path = Path("output"),
    write_csv: bool = True,
) -> dict[str, pd.DataFrame]:
    results = {}

    for ablation_dir in sorted(output_root.glob("ablations_*")):
        if not ablation_dir.is_dir():
            continue

        dataset_dirs = sorted(
            p for p in ablation_dir.iterdir()
            if p.is_dir() and p.name != "generated_configs"
        )

        for dataset_dir in dataset_dirs:
            key = f"{ablation_dir.name}/{dataset_dir.name}"
            results[key] = read_dataset_results(dataset_dir, write_csv=write_csv)

    return results


# Aggregate metrics files are preferred when present. The expected project layout is:
#
#     output/ablations_*/checkpoint_eval_metrics.csv
#     output/ablations_*/checkpoint_eval_metrics.jsonl
#     output/ablations_*/ablation_metrics.csv
#     output/ablations_*/ablation_metrics.jsonl
#
# The notebook also accepts the same exported files next to the notebook for
# ad-hoc analysis. When both CSV and JSONL exports exist in the same directory,
# CSV is preferred to avoid loading duplicate rows.
AGGREGATE_METRICS_FILES = [
    Path("checkpoint_eval_metrics.csv"),
    Path("checkpoint_eval_metrics.jsonl"),
    Path("ablation_metrics.csv"),
    Path("ablation_metrics.jsonl"),
]


def read_metrics_file(path: Path) -> pd.DataFrame:
    """Read one metrics export, supporting both CSV and JSONL/NDJSON."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path, skipinitialspace=True)

    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)

    if suffix == ".json":
        return pd.read_json(path)

    raise ValueError(
        f"Unsupported metrics file type for {path}. "
        "Expected .csv, .jsonl, .ndjson, or .json."
    )


def infer_dataset_key_from_path(value) -> str | pd._libs.missing.NAType:
    if is_missing(value):
        return pd.NA

    parts = Path(str(value)).parts

    # Typical path shape:
    # .../output/ablations_fixed_longer_newer/bouncingballs/<variant>/...
    for i, part in enumerate(parts):
        if part == "output" and i + 2 < len(parts):
            return f"{parts[i + 1]}/{parts[i + 2]}"

    # Fallback for paths that start at the ablations_* directory.
    for i, part in enumerate(parts):
        if part.startswith("ablations_") and i + 1 < len(parts):
            return f"{part}/{parts[i + 1]}"

    return pd.NA


def infer_variant_parts_from_path(value) -> dict:
    if is_missing(value):
        return {}

    candidate = Path(str(value)).name

    # If the value is a checkpoint file path, the variant folder is the parent.
    if candidate.startswith("chkpnt"):
        candidate = Path(str(value)).parent.name

    try:
        return parse_variant_dirname(candidate)
    except ValueError:
        return {}


def fill_missing_from_variant_path(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    source_col = None
    for col in ["model_path", "checkpoint_path", "generated_config_path"]:
        if col in df.columns:
            source_col = col
            break

    if source_col is None:
        return df

    parsed_rows = df[source_col].apply(infer_variant_parts_from_path)

    for col in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess", "use_usplat", "variant_name"]:
        if col not in df.columns:
            df[col] = pd.NA

        inferred = parsed_rows.apply(lambda d: d.get(col, pd.NA))
        df[col] = df[col].where(df[col].notna(), inferred)

    return df


def normalize_metrics_dataframe(df: pd.DataFrame, *, source_path: Path | None = None) -> pd.DataFrame:
    """Normalize aggregate CSV/JSONL metrics to the columns used by the notebook."""
    df = clean_string_columns(df)
    df = fill_missing_from_variant_path(df)

    if "dataset_key" not in df.columns:
        df["dataset_key"] = pd.NA

    path_cols = [c for c in ["model_path", "checkpoint_path", "generated_config_path"] if c in df.columns]
    for col in path_cols:
        inferred = df[col].apply(infer_dataset_key_from_path)
        df["dataset_key"] = df["dataset_key"].where(df["dataset_key"].notna(), inferred)

    if "dataset_key" in df.columns and "scene_name" in df.columns:
        if source_path is not None and Path(source_path).parent.name.startswith("ablations_"):
            fallback_prefix = Path(source_path).parent.name
        else:
            fallback_prefix = "uploaded_metrics"

        fallback_key = fallback_prefix + "/" + df["scene_name"].astype("string")
        df["dataset_key"] = df["dataset_key"].where(df["dataset_key"].notna(), fallback_key)

    default_values = {
        "matrix_preset": MATRIX_PRESET,
        "use_usplat": USE_USPLAT_DEFAULT,
        "dropout": DROPOUT_DEFAULT,
        "metrics_csv_path": pd.NA,
        "metrics_jsonl_path": pd.NA,
        "created_at": datetime.now().replace(microsecond=0).isoformat(),
    }

    for col, value in default_values.items():
        if col not in df.columns:
            df[col] = value
        else:
            df[col] = df[col].where(df[col].notna(), value)

    if source_path is not None:
        source_path = Path(source_path)
        if source_path.suffix.lower() == ".csv":
            df["metrics_csv_path"] = df["metrics_csv_path"].where(
                df["metrics_csv_path"].notna(),
                str(source_path.resolve()),
            )
        elif source_path.suffix.lower() in {".jsonl", ".ndjson", ".json"}:
            df["metrics_jsonl_path"] = df["metrics_jsonl_path"].where(
                df["metrics_jsonl_path"].notna(),
                str(source_path.resolve()),
            )

    if "checkpoint_filename" not in df.columns and "checkpoint_path" in df.columns:
        df["checkpoint_filename"] = df["checkpoint_path"].apply(
            lambda value: Path(str(value)).name if not is_missing(value) else pd.NA
        )

    if "checkpoint_name_iteration" not in df.columns:
        df["checkpoint_name_iteration"] = pd.NA

    if "eval_checkpoint_iteration" in df.columns:
        df["checkpoint_name_iteration"] = df["checkpoint_name_iteration"].where(
            df["checkpoint_name_iteration"].notna(),
            df["eval_checkpoint_iteration"],
        )

    if "training_wall_clock_sec_at_checkpoint" not in df.columns:
        df["training_wall_clock_sec_at_checkpoint"] = pd.NA

    if "training_wall_clock_sec" in df.columns:
        df["training_wall_clock_sec_at_checkpoint"] = df[
            "training_wall_clock_sec_at_checkpoint"
        ].where(
            df["training_wall_clock_sec_at_checkpoint"].notna(),
            df["training_wall_clock_sec"],
        )

    if "variant_index" not in df.columns:
        group_cols = [c for c in ["dataset_key", "run_hash", "variant_name", "model_path"] if c in df.columns]
        if group_cols:
            df["variant_index"] = df.groupby(group_cols, dropna=False).ngroup()
        else:
            df["variant_index"] = pd.NA

    df = add_mb_columns(df)

    for col in ["checkpoint_name_iteration", "eval_checkpoint_iteration", *METRIC_COLS]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def first_existing_metrics_file(directory: Path) -> Path | None:
    """Return the preferred aggregate metrics file in a directory, if any."""
    directory = Path(directory)

    for rel_path in AGGREGATE_METRICS_FILES:
        candidate = directory / rel_path.name
        if candidate.exists():
            return candidate

    return None


def discover_aggregate_metrics_files(
    *,
    output_root: Path = Path("output"),
    search_dir: Path = Path("."),
) -> list[Path]:
    """
    Find aggregate metrics exports.

    Preferred project layout:
        output/ablations_*/checkpoint_eval_metrics.csv
        output/ablations_*/checkpoint_eval_metrics.jsonl
        output/ablations_*/ablation_metrics.csv
        output/ablations_*/ablation_metrics.jsonl

    Fallback ad-hoc layout:
        ./checkpoint_eval_metrics.csv, ./checkpoint_eval_metrics.jsonl, etc.
    """
    output_root = Path(output_root)
    search_dir = Path(search_dir)
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add(path: Path | None) -> None:
        if path is None:
            return

        resolved = path.resolve()
        if resolved in seen:
            return

        seen.add(resolved)
        candidates.append(path)

    # Project layout: one aggregate export at the root of each ablations_* run.
    if output_root.exists():
        for ablation_dir in sorted(output_root.glob("ablations_*")):
            if ablation_dir.is_dir():
                add(first_existing_metrics_file(ablation_dir))

    # Fallback for uploaded/exported files placed next to this notebook.
    if not candidates:
        add(first_existing_metrics_file(search_dir))

    return candidates


def load_aggregate_metrics(
    paths: list[Path] | None = None,
    *,
    search_dir: Path = Path("."),
    output_root: Path = Path("output"),
) -> pd.DataFrame | None:
    """Load available aggregate metrics CSV/JSONL exports and concatenate them."""
    if paths is None:
        paths = discover_aggregate_metrics_files(
            output_root=output_root,
            search_dir=search_dir,
        )
    else:
        paths = [Path(p) for p in paths]

    loaded: list[pd.DataFrame] = []

    for path in paths:
        path = Path(path)
        candidate = path if path.is_absolute() else search_dir / path

        if not candidate.exists():
            continue

        print(f"Reading aggregate metrics file: {candidate}")
        loaded.append(
            normalize_metrics_dataframe(
                read_metrics_file(candidate),
                source_path=candidate,
            )
        )

    if not loaded:
        return None

    return pd.concat(loaded, ignore_index=True, sort=False)


BEST_CHECKPOINT_NAME_PATTERN = r"(?:chkpnt|checkpoint).*best|best.*(?:chkpnt|checkpoint)"
BEST_CHECKPOINT_BOOL_COLS = [
    "is_best_checkpoint",
    "best_checkpoint",
    "best_chkpnt",
    "is_best_chkpnt",
]
BEST_CHECKPOINT_NAME_COLS = [
    "checkpoint_filename",
    "checkpoint_path",
]


def boolish_series(s: pd.Series) -> pd.Series:
    """Convert common truthy values to booleans without treating NA as true."""
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)

    return (
        s.astype("string")
        .str.strip()
        .str.lower()
        .isin({"1", "true", "yes", "y", "best", "best_chkpnt", "chkpnt_best"})
    )


def best_checkpoint_mask(df: pd.DataFrame) -> pd.Series:
    """Rows whose checkpoint is explicitly marked as the best checkpoint."""
    mask = pd.Series(False, index=df.index)

    for col in BEST_CHECKPOINT_BOOL_COLS:
        if col in df.columns:
            mask |= boolish_series(df[col])

    for col in BEST_CHECKPOINT_NAME_COLS:
        if col not in df.columns:
            continue

        values = df[col].astype("string").fillna("")

        if col.endswith("path"):
            values = values.apply(lambda value: Path(str(value)).name if value else "")

        mask |= values.str.contains(
            BEST_CHECKPOINT_NAME_PATTERN,
            case=False,
            regex=True,
            na=False,
        )

    return mask


def select_best_checkpoint_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only best-checkpoint rows and one row per run.

    The analysis should compare variants at the checkpoint selected by training,
    not at the numerically latest checkpoint. This recognizes both
    chkpnt_best.pth and best_chkpnt.pth naming conventions, plus boolean marker
    columns such as best_chkpnt=True.
    """
    df = df.copy()

    if df.empty:
        return df

    mask = best_checkpoint_mask(df)
    best_rows = df.loc[mask].copy()

    if best_rows.empty:
        checked_cols = [
            c
            for c in [*BEST_CHECKPOINT_BOOL_COLS, *BEST_CHECKPOINT_NAME_COLS]
            if c in df.columns
        ]
        raise ValueError(
            "No best-checkpoint rows found. Expected checkpoint_filename or "
            "checkpoint_path to contain chkpnt_best.pth or best_chkpnt.pth, "
            "or a boolean marker column such as best_chkpnt=True. "
            f"Checked columns: {checked_cols or 'none'}."
        )

    group_cols = [c for c in ["dataset_key", "run_hash"] if c in best_rows.columns]

    if not group_cols:
        group_cols = [c for c in ["model_path", "variant_name", "scene_name"] if c in best_rows.columns]

    if group_cols:
        sort_cols = []

        if "status" in best_rows.columns:
            best_rows["_status_ok"] = best_rows["status"].astype("string").str.lower().eq("ok")
            sort_cols.append("_status_ok")

        if "eval_checkpoint_iteration" in best_rows.columns:
            sort_cols.append("eval_checkpoint_iteration")

        if sort_cols:
            best_rows = best_rows.sort_values(
                sort_cols,
                ascending=[False] + [True] * (len(sort_cols) - 1),
                kind="stable",
            )

        before = len(best_rows)
        best_rows = best_rows.drop_duplicates(group_cols, keep="first")
        duplicate_count = before - len(best_rows)
        if duplicate_count:
            print(f"Dropped {duplicate_count} duplicate best-checkpoint rows using {group_cols}.")

        if "_status_ok" in best_rows.columns:
            best_rows = best_rows.drop(columns="_status_ok")

    print(f"Selected {len(best_rows)} best-checkpoint rows from {len(df)} total rows.")
    return best_rows.reset_index(drop=True)


def apply_dataset_overrides_to_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "dataset_key" not in df.columns:
        return df

    for key, overrides in DATASET_OVERRIDES.items():
        mask = df["dataset_key"].eq(key)
        if not mask.any():
            continue

        for col, value in overrides.items():
            df.loc[mask, col] = value

    for key in INFER_USPLAT_KEYS:
        mask = df["dataset_key"].eq(key)
        if not mask.any():
            continue

        df.loc[mask, "use_usplat"] = df.loc[mask].apply(
            infer_use_usplat,
            axis=1,
        )

    return df


def apply_drop_rules_to_frame(
    df: pd.DataFrame,
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    if not drop_rules or "dataset_key" not in df.columns:
        return df.copy()

    keep_mask = pd.Series(True, index=df.index)

    for key, col, value in drop_rules:
        if col not in df.columns:
            print(f"{key}: skip missing drop column: {col}")
            continue

        mask = df["dataset_key"].eq(key) & df[col].astype(str).str.lower().eq(str(value).lower())
        dropped = int(mask.sum())
        total = int(df["dataset_key"].eq(key).sum())
        frac = dropped / total if total else 0.0
        print(f"{key}: dropped {dropped}/{total} rows ({frac:.1%})")
        keep_mask &= ~mask

    return df.loc[keep_mask].copy()


def infer_use_usplat(row: pd.Series) -> bool:
    tokens = []

    for col in ["variant_name", "model_path", "generated_config_path"]:
        value = row.get(col)

        if pd.isna(value):
            continue

        value = str(value)

        if col == "model_path":
            tokens.extend(Path(value).name.split("--"))
        elif col == "generated_config_path":
            tokens.extend(Path(value).stem.split("__"))
        else:
            tokens.extend(value.split("__"))

    return "use_usplat" in tokens


def apply_dataset_overrides(dataset_results: dict[str, pd.DataFrame]) -> None:
    for key, overrides in DATASET_OVERRIDES.items():
        if key not in dataset_results:
            print(f"Skipping missing override dataset: {key}")
            continue

        for col, value in overrides.items():
            dataset_results[key][col] = value

    for key in INFER_USPLAT_KEYS:
        if key not in dataset_results:
            print(f"Skipping missing inferred usplat dataset: {key}")
            continue

        dataset_results[key]["use_usplat"] = dataset_results[key].apply(
            infer_use_usplat,
            axis=1,
        )


def merge_dataset_results(
    dataset_results: dict[str, pd.DataFrame],
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    rules_by_key: dict[str, list[tuple[str, object]]] = {}

    for key, col, value in drop_rules or []:
        rules_by_key.setdefault(key, []).append((col, value))

    merged = []

    for key, df in dataset_results.items():
        df = df.copy()
        original_len = len(df)

        drop_mask = pd.Series(False, index=df.index)

        for col, value in rules_by_key.get(key, []):
            if col not in df.columns:
                print(f"{key}: skip missing drop column: {col}")
                continue

            drop_mask |= df[col].astype(str).str.lower().eq(str(value).lower())

        dropped = int(drop_mask.sum())
        kept = df.loc[~drop_mask].copy()

        frac = dropped / original_len if original_len else 0.0
        print(f"{key}: dropped {dropped}/{original_len} rows ({frac:.1%})")

        kept.insert(0, "dataset_key", key)
        merged.append(kept)

    return pd.concat(merged, ignore_index=True) if merged else pd.DataFrame()


ABLATION_BOOL_COLS = [
    "isotropy",
    "use_usplat",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
]

CHECKPOINT_COLS = [
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
]

MODEL_SPEC_COLS = [
    "run_hash",
    "dataset_key",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    *ABLATION_BOOL_COLS,
    *CHECKPOINT_COLS,
]

ABLATION_COLS = [
    "scene_name",
    *ABLATION_BOOL_COLS,
    "eval_checkpoint_iteration",
]

RADAR_SPECS = {
    "+PSNR": {"col": "psnr", "higher_is_better": True},
    "+SSIM": {"col": "ssim", "higher_is_better": True},
    "-LPIPS": {"col": "lpips", "higher_is_better": False},
    "+FPS": {"col": "render_fps", "higher_is_better": True},
    "-VRAM": {"col": "peak_eval_vram_mb", "higher_is_better": False},
    "-Size": {"col": "checkpoint_size_mb", "higher_is_better": False},
    "-Time": {"col": "training_wall_clock_sec_at_checkpoint", "higher_is_better": False},
    "-Gaussians": {"col": "final_gaussian_count", "higher_is_better": False},
}

METRIC_COLS = [spec["col"] for spec in RADAR_SPECS.values()]

BOOLEAN_LABEL_PAIRS = {
    "isotropy": ("isotropy", "anisotropy"),
    "appearance": ("appearance", "no appearance"),
    "sorting": ("sorting", "no sorting"),
    "pruning": ("pruning", "no pruning"),
    "dropout": ("dropout", "no dropout"),
    "ess": ("ess", "no ess"),
    "use_usplat": ("usplat", "no usplat"),
    "uncertain": ("uncertain", "no uncertain"),
}

BOOLEAN_LABEL_MAP = {
    col: {
        True: true_label,
        False: false_label,
    }
    for col, (true_label, false_label) in BOOLEAN_LABEL_PAIRS.items()
}


aggregate_df = load_aggregate_metrics()

if aggregate_df is not None:
    final_df = apply_dataset_overrides_to_frame(aggregate_df)
    final_df = apply_drop_rules_to_frame(final_df, drop_rules=DROP_RULES)
else:
    dataset_results = read_all_dataset_results()
    if not dataset_results:
        raise FileNotFoundError(
            "No aggregate metrics file was found and no output/ablations_* "
            "directory tree was found. Expected aggregate exports at "
            "output/ablations_*/checkpoint_eval_metrics.csv, "
            "output/ablations_*/checkpoint_eval_metrics.jsonl, "
            "output/ablations_*/ablation_metrics.csv, or "
            "output/ablations_*/ablation_metrics.jsonl. The same files may also "
            "be placed next to this notebook for ad-hoc analysis."
        )

    apply_dataset_overrides(dataset_results)

    final_df = merge_dataset_results(
        dataset_results,
        drop_rules=DROP_RULES,
    )

final_df = select_best_checkpoint_rows(final_df)

keep_cols = [c for c in MODEL_SPEC_COLS + METRIC_COLS if c in final_df.columns]
print("Keep:", keep_cols)

df_clean = final_df[keep_cols].copy()

print("\nShape: ", df_clean.shape)

Reading aggregate metrics file: output/ablations_fixed_longer_newer/checkpoint_eval_metrics.csv
ablations_no_usplat_dropout_off/trex: dropped 0/0 rows (0.0%)
Selected 128 best-checkpoint rows from 1116 total rows.
Keep: ['run_hash', 'dataset_key', 'scene_name', 'variant_name', 'variant_index', 'matrix_preset', 'isotropy', 'use_usplat', 'appearance', 'sorting', 'pruning', 'dropout', 'ess', 'checkpoint_index', 'checkpoint_filename', 'checkpoint_path', 'checkpoint_name_iteration', 'eval_checkpoint_iteration', 'psnr', 'ssim', 'lpips', 'render_fps', 'peak_eval_vram_mb', 'checkpoint_size_mb', 'training_wall_clock_sec_at_checkpoint', 'final_gaussian_count']

Shape:  (128, 26)


In [42]:
df_clean = df_clean[df_clean["dataset_key"].str.startswith("ablations_fixed_longer_newer")]

# The loader has already restricted df_clean to best-checkpoint rows.
df_clean = df_clean.reset_index(drop=True)

## Memory

In [43]:
import pandas as pd


def make_typst_ablation_table(
    ablations_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    psnr_bad_below: float = -1e100,
    ssim_bad_below: float = -1e100,
    lpips_bad_above: float = 1e100,
) -> str:
    def best_checkpoint_per_run(df: pd.DataFrame) -> pd.DataFrame:
        marker_cols = [
            c
            for c in [*BEST_CHECKPOINT_BOOL_COLS, *BEST_CHECKPOINT_NAME_COLS]
            if c in df.columns
        ]

        if marker_cols:
            return select_best_checkpoint_rows(df)

        # The loader normally filters to best-checkpoint rows before this helper is
        # called. If checkpoint marker columns were intentionally omitted from a
        # narrow dataframe, allow it only when there is already at most one row per run.
        if "run_hash" in df.columns and df["run_hash"].duplicated().any():
            raise ValueError(
                "Cannot choose best checkpoint because this dataframe has duplicate "
                "run_hash values but no checkpoint_filename/checkpoint_path or "
                "best_chkpnt marker columns."
            )

        return df.reset_index(drop=True)

    def esc(x):
        if pd.isna(x):
            return ""
        return (
            str(x)
            .replace("\\", "\\\\")
            .replace("[", "\\[")
            .replace("]", "\\]")
            .replace("#", "\\#")
        )

    def short(x):
        mapping = {
            "anisotropic": "aniso",
            "isotropic": "iso",
            "rgb": "RGB",
            "sh3": "SH3",
            "sort": "sort",
            "no_pruning": "none",
            "interleaved_prune_densify": "prune+dense",
            "early_init_pruning": "early",
            "final_pruning": "final",
            "final": "final",
            "sort_free": "sortfree",
            "dropout": "yes",
            "yes_dropout": "yes",
            "no_dropout": "no",
            "ess": "yes",
            "no_ess": "no",
            True: "yes",
            False: "no",
        }
        return mapping.get(x, x)

    def fmt(col, x):
        if pd.isna(x):
            return ""

        if col == "psnr":
            return f"{x:.2f}"
        if col in ["ssim", "lpips"]:
            return f"{x:.3f}"
        if col == "render_fps":
            return f"{x:.0f}"
        if col == "peak_eval_vram_mb":
            return f"{x:.0f}"
        if col == "checkpoint_size_mb":
            return f"{x:.1f}"
        if col == "final_gaussian_count":
            return f"{x / 1000:.1f}"
        if col == "training_wall_clock_sec_at_checkpoint":
            return f"{x / 60:.0f}m"
        if col == "eval_checkpoint_iteration":
            return f"{x / 1000:.0f}"

        return str(short(x))

    def is_bad_metric(col, x) -> bool:
        if pd.isna(x):
            return False

        x = float(x)

        if col == "psnr":
            return x < psnr_bad_below

        if col == "ssim":
            return x < ssim_bad_below

        if col == "lpips":
            return x > lpips_bad_above

        return False

    def ablation_color(col, x):
        v = str(short(x)).lower()

        if v in ["yes", "true"]:
            return "#d9efdf"

        if v in ["no", "false", "none"]:
            return "#f2f2f2"

        if v in ["iso", "isotropic"]:
            return "#e3f2fd"

        if v in ["aniso", "anisotropic"]:
            return "#fff4bf"

        if v in ["rgb"]:
            return "#fce4ec"

        if v in ["sh3"]:
            return "#ede7f6"

        if v in ["sort"]:
            return "#d9efdf"

        if v in ["sortfree", "sort_free"]:
            return "#f2f2f2"

        if v in ["prune+dense"]:
            return "#ffe0b2"

        return "#ffffff"

    def quantile_color(q: float) -> str:
        """
        Lower quantiles are better for Gaussians and train time.

        0.0 -> best shade
        1.0 -> worst shade
        """
        if pd.isna(q):
            return "#ffffff"

        if q <= 0.20:
            return "#d9efdf"
        if q <= 0.40:
            return "#eaf6e9"
        if q <= 0.60:
            return "#fff4bf"
        if q <= 0.80:
            return "#ffe0b2"

        return "#ffd6d6"

    def column_quantile_colors(df: pd.DataFrame, col: str) -> dict[int, str]:
        """
        Returns row-index -> color for a numeric column.

        Uses rank-based quantiles so ties get the same averaged quantile.
        Lower values receive greener shades.
        """
        if col not in df.columns:
            return {}

        s = pd.to_numeric(df[col], errors="coerce")
        valid = s.dropna()

        if valid.empty:
            return {}

        if len(valid) == 1 or valid.nunique(dropna=True) == 1:
            return {idx: "#d9efdf" for idx in valid.index}

        ranks = valid.rank(method="average")
        quantiles = (ranks - 1) / (len(valid) - 1)

        return {
            idx: quantile_color(q)
            for idx, q in quantiles.items()
        }

    def worst_25_quantile_indices(
        df: pd.DataFrame,
        col: str,
        higher_is_worse: bool,
    ) -> set[int]:
        """
        Returns indices in the worst 25% for a numeric column.

        For FPS:
          lower values are worse.

        For VRAM and MB:
          higher values are worse.
        """
        if col not in df.columns:
            return set()

        s = pd.to_numeric(df[col], errors="coerce")
        valid = s.dropna()

        if valid.empty:
            return set()

        if len(valid) == 1 or valid.nunique(dropna=True) == 1:
            return set()

        ranks = valid.rank(method="average")
        quantiles = (ranks - 1) / (len(valid) - 1)

        if higher_is_worse:
            return set(quantiles[quantiles >= 0.75].index)

        return set(quantiles[quantiles <= 0.25].index)

    def render_one_table(df: pd.DataFrame, caption: str) -> str:
        df = df.copy().reset_index(drop=True)

        ablation_candidates = [
            "isotropy",
            "sorting",
            "appearance",
            "pruning",
            "dropout",
            "ess",
            "use_usplat",
        ]

        ablation_cols = []
        for c in ablation_candidates:
            if c in df.columns and df[c].nunique(dropna=False) > 1:
                ablation_cols.append(c)

        if ablation_cols:
            df = (
                df.sort_values(
                    by=ablation_cols,
                    key=lambda s: s.map(short).astype(str),
                    kind="stable",
                    na_position="last",
                )
                .reset_index(drop=True)
            )

        metric_cols = [
            c for c in [
                "eval_checkpoint_iteration",
                "psnr",
                "ssim",
                "lpips",
                "render_fps",
                "peak_eval_vram_mb",
                "checkpoint_size_mb",
                "final_gaussian_count",
                "training_wall_clock_sec_at_checkpoint",
            ]
            if c in df.columns
        ]

        best_metric_cols = {
            "psnr": "max",
            "ssim": "max",
            "lpips": "min",
            "render_fps": "max",
            "peak_eval_vram_mb": "min",
            "checkpoint_size_mb": "min",
        }

        quantile_shaded_cols = {
            "final_gaussian_count",
            "training_wall_clock_sec_at_checkpoint",
        }

        quantile_colors = {
            c: column_quantile_colors(df, c)
            for c in quantile_shaded_cols
            if c in df.columns
        }

        worst_25_cols = {
            "render_fps": False,              # lower FPS is worse
            "peak_eval_vram_mb": True,        # higher VRAM is worse
            "checkpoint_size_mb": True,       # higher MB is worse
        }

        worst_25_indices = {
            c: worst_25_quantile_indices(df, c, higher_is_worse)
            for c, higher_is_worse in worst_25_cols.items()
            if c in df.columns
        }

        row_has_bad_metric = df.apply(
            lambda row: any(is_bad_metric(c, row[c]) for c in metric_cols),
            axis=1,
        )

        acceptable_df = df.loc[~row_has_bad_metric]

        best_values = {}
        for c, direction in best_metric_cols.items():
            if c in acceptable_df.columns:
                s = pd.to_numeric(acceptable_df[c], errors="coerce")
                best_values[c] = s.max() if direction == "max" else s.min()

        names = {
            "isotropy": "Isotropy",
            "sorting": "Sorting",
            "appearance": "Color",
            "pruning": "Prune",
            "dropout": "Drop",
            "ess": "ESS",
            "use_usplat": "USplat",
            "eval_checkpoint_iteration": "Iter  ×10³",
            "psnr": "PSNR ↑",
            "ssim": "SSIM ↑",
            "lpips": "LPIPS ↓",
            "render_fps": "FPS ↑",
            "peak_eval_vram_mb": "VRAM ↓",
            "checkpoint_size_mb": "MB ↓",
            "final_gaussian_count": "Gauss k ↓",
            "training_wall_clock_sec_at_checkpoint": "Train m ↓",
        }

        columns = ablation_cols + metric_cols

        n_ablation = len(ablation_cols)
        n_visual = len(metric_cols)

        col_widths = ["0.7fr"] * n_ablation + ["0.7fr"] * n_visual

        lines = []

        lines.append("#figure(")
        lines.append(f"  caption: [{esc(caption)}],")
        lines.append("  [")
        lines.append("    #set text(size: 6.5pt)")
        lines.append("")
        lines.append('    #let dark = rgb("#2c3e50")')
        lines.append('    #let section = rgb("#f2f2f2")')
        lines.append('    #let border = rgb("#cccccc")')
        lines.append('    #let bad = rgb("#ffd6d6")')
        lines.append('    #let bad-text = rgb("#b00020")')
        lines.append("")
        lines.append("    #table(")
        lines.append(f"      columns: ({', '.join(col_widths)}),")
        lines.append("      stroke: border,")
        lines.append("      inset: 3pt,")
        lines.append("      align: center + horizon,")
        lines.append("")
        lines.append("      table.header(")
        lines.append("        repeat: true,")

        if n_ablation > 0:
            lines.append(
                f"        table.cell(colspan: {n_ablation}, fill: section)"
                '[#text(weight: "bold")[ABLATIONS]],'
            )

        if n_visual > 0:
            lines.append(
                f"        table.cell(colspan: {n_visual}, fill: section)"
                '[#text(weight: "bold")[VISUAL]],'
            )

        for c in columns:
            lines.append(
                f"        table.cell(fill: dark)"
                f'[#text(fill: white, weight: "bold")[{esc(names.get(c, c))}]],'
            )

        lines.append("      ),")
        lines.append("")

        for idx, row in df.iterrows():
            cells = []

            for c in columns:
                value = esc(fmt(c, row[c]))

                if c in ablation_cols:
                    color = ablation_color(c, row[c])
                    cells.append(f'table.cell(fill: rgb("{color}"))[{value}]')

                elif is_bad_metric(c, row[c]):
                    cells.append(
                        f'table.cell(fill: bad)[#text(fill: bad-text, weight: "bold")[{value}]]'
                    )

                elif c in worst_25_indices and idx in worst_25_indices[c]:
                    cells.append(
                        f'table.cell(fill: bad)[#text(fill: bad-text, weight: "bold")[{value}]]'
                    )

                elif (
                    not row_has_bad_metric.loc[idx]
                    and c in best_values
                    and pd.notna(row[c])
                    and row[c] == best_values[c]
                ):
                    cells.append(f'table.cell(fill: rgb("#d9efdf"))[{value}]')

                elif c in quantile_colors and idx in quantile_colors[c]:
                    color = quantile_colors[c][idx]
                    cells.append(f'table.cell(fill: rgb("{color}"))[{value}]')

                else:
                    cells.append(f"[{value}]")

            lines.append("        " + ", ".join(cells) + ",")

        lines.append("    )")
        lines.append("  ],")
        lines.append(")")

        return "\n".join(lines)

    ablations_df = best_checkpoint_per_run(ablations_df)
    metrics_df = best_checkpoint_per_run(metrics_df)

    df = ablations_df.merge(
        metrics_df,
        on="run_hash",
        how="left",
        suffixes=("", "_metric"),
    )

    typst_preamble = "#show figure: set block(breakable: true)"

    if "scene_name" not in df.columns:
        return typst_preamble + "\n\n" + render_one_table(df, "Ablations")

    tables = []
    for scene_value, scene_df in df.groupby("scene_name", sort=False, dropna=False):
        tables.append(render_one_table(scene_df, scene_value))

    return typst_preamble + "\n\n" + "\n\n".join(tables)

In [44]:
from pathlib import Path

ablation_columns = [
    "run_hash",
    "scene_name",
    "variant_name",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "checkpoint_filename",
    "checkpoint_path",
    "eval_checkpoint_iteration",
]

metric_columns = [
    "run_hash",
    "checkpoint_filename",
    "checkpoint_path",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
]

out_files = []


df = df_clean

ablations_df = df[ablation_columns].copy()
metrics_df = df[metric_columns].copy()

typst_table = make_typst_ablation_table(
    ablations_df,
    metrics_df,
    psnr_bad_below=20,
    ssim_bad_below=0.9,
    lpips_bad_above=0.1,
)

print(typst_table)

Selected 128 best-checkpoint rows from 128 total rows.
Selected 128 best-checkpoint rows from 128 total rows.
#show figure: set block(breakable: true)

#figure(
  caption: [bouncingballs],
  [
    #set text(size: 6.5pt)

    #let dark = rgb("#2c3e50")
    #let section = rgb("#f2f2f2")
    #let border = rgb("#cccccc")
    #let bad = rgb("#ffd6d6")
    #let bad-text = rgb("#b00020")

    #table(
      columns: (0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr),
      stroke: border,
      inset: 3pt,
      align: center + horizon,

      table.header(
        repeat: true,
        table.cell(colspan: 5, fill: section)[#text(weight: "bold")[ABLATIONS]],
        table.cell(colspan: 9, fill: section)[#text(weight: "bold")[VISUAL]],
        table.cell(fill: dark)[#text(fill: white, weight: "bold")[Isotropy]],
        table.cell(fill: dark)[#text(fill: white, weight: "bold")[Color]],
        table.cell(fill: dark)[#text(fill: white, weight: "bol

In [45]:
import numpy as np
import pandas as pd


LOWER_IS_BETTER = {
    "lpips",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
}

METRIC_LABELS = {
    "psnr": "PSNR",
    "ssim": "SSIM",
    "lpips": "LPIPS",
    "render_fps": "render FPS",
    "peak_eval_vram_mb": "eval VRAM",
    "final_gaussian_count": "Gaussian count",
    "checkpoint_size_mb": "checkpoint size",
    "training_wall_clock_sec_at_checkpoint": "training time",
}

METRIC_EXCLUDE = {
    "run_hash",
    "checkpoint_index",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
    "variant_index",
}


def cohens_d(x: pd.Series, y: pd.Series) -> float:
    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_var = (
        ((n_x - 1) * x.var(ddof=1) + (n_y - 1) * y.var(ddof=1))
        / (n_x + n_y - 2)
    )

    pooled_std = np.sqrt(pooled_var)
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else np.nan


def hedges_g(x: pd.Series, y: pd.Series) -> float:
    """
    Hedges' g = Cohen's d with small-sample bias correction.
    Independent-groups version.
    """
    n_x = len(x)
    n_y = len(y)
    df = n_x + n_y - 2

    if n_x < 2 or n_y < 2 or df <= 1:
        return np.nan

    d = cohens_d(x, y)

    if pd.isna(d):
        return np.nan

    correction = 1 - (3 / (4 * df - 1))
    return correction * d


def robust_g(x: pd.Series, y: pd.Series) -> float:
    """
    Robust standardized median difference.

    This is not technically Hedges' g, but is kept as a robust companion
    to Hedges' g, analogous to your previous robust_d.
    """
    x_med = x.median()
    y_med = y.median()

    pooled_mad = np.nanmean([
        (x - x_med).abs().median(),
        (y - y_med).abs().median(),
    ])

    return (x_med - y_med) / (1.4826 * pooled_mad) if pooled_mad > 0 else np.nan


def analyze_ablation_effects_hedges_g(
    df: pd.DataFrame,
    ablation_columns: list[str],
    metric_columns: list[str],
    *,
    ablation_group_columns: list[str] | None = None,
    min_abs_effect: float = 0.2,
    return_all: bool = False,
) -> pd.DataFrame:
    df = df.copy()

    metrics = [
        c for c in metric_columns
        if c in df.columns and c not in METRIC_EXCLUDE
    ]

    for c in metrics:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    metrics = [c for c in metrics if df[c].notna().any()]

    if ablation_group_columns is None:
        ablation_group_columns = [
            c for c in ablation_columns
            if c in df.columns
            and c not in {
                "run_hash",
                "scene_name",
                "variant_name",
                "eval_checkpoint_iteration",
            }
        ]

    rows = []

    for ablation_col in ablation_group_columns:
        for ablation_value, group in df.groupby(ablation_col, dropna=False):
            other = (
                df[df[ablation_col].notna()]
                if pd.isna(ablation_value)
                else df[df[ablation_col] != ablation_value]
            )

            if other.empty:
                continue

            for metric in metrics:
                x = group[metric].dropna()
                y = other[metric].dropna()

                if len(x) < 2 or len(y) < 2:
                    continue

                g = hedges_g(x, y)
                rg = robust_g(x, y)

                effects = [abs(v) for v in [g, rg] if pd.notna(v)]
                if not effects:
                    continue

                abs_effect = max(effects)

                if not return_all and abs_effect < min_abs_effect:
                    continue

                sign = -1 if metric in LOWER_IS_BETTER else 1

                raw_delta = x.mean() - y.mean()
                benefit_delta = sign * raw_delta
                benefit_g = sign * g if pd.notna(g) else np.nan
                benefit_rg = sign * rg if pd.notna(rg) else np.nan

                rows.append({
                    "ablation_column": ablation_col,
                    "ablation_value": ablation_value,
                    "metric": metric,
                    "n": len(x),
                    "other_n": len(y),

                    "group_mean": x.mean(),
                    "other_mean": y.mean(),
                    "delta_mean": raw_delta,
                    "pct_delta_mean": 100 * raw_delta / abs(y.mean()) if y.mean() else np.nan,

                    "group_median": x.median(),
                    "other_median": y.median(),
                    "delta_median": x.median() - y.median(),

                    "hedges_g": g,
                    "robust_effect": rg,
                    "benefit_delta_mean": benefit_delta,
                    "benefit_effect": benefit_g,
                    "benefit_robust_effect": benefit_rg,
                    "abs_effect": abs_effect,

                    "interpretation": (
                        "better" if benefit_delta > 0
                        else "worse" if benefit_delta < 0
                        else "same"
                    ),
                })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["abs_effect", "ablation_column", "metric"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


def fmt_value(metric: str, x: float) -> str:
    if pd.isna(x):
        return "nan"
    if metric in {"ssim", "lpips"}:
        return f"{x:.4f}"
    if metric == "psnr":
        return f"{x:.2f}"
    if metric == "render_fps":
        return f"{x:.1f}"
    if metric == "final_gaussian_count":
        return f"{x:,.0f}"
    return f"{x:.1f}"


def effect_strength(g: float) -> str:
    if g >= 2.0:
        return "very large"
    if g >= 1.0:
        return "large"
    if g >= 0.5:
        return "moderate"
    return "small"


def metric_list(gdf: pd.DataFrame, interpretation: str) -> str:
    rows = (
        gdf[gdf["interpretation"].eq(interpretation)]
        .sort_values("abs_g", ascending=False)
    )

    if rows.empty:
        return "none"

    parts = []

    for _, r in rows.iterrows():
        sign = "+" if r["benefit_g"] > 0 else ""
        parts.append(f"{r['metric_label']} ({sign}{r['benefit_g']:.2f})")

    return ", ".join(parts)


def ablation_effects_to_text_hedges_g(
    table: pd.DataFrame,
    *,
    top_n: int = 40,
    min_abs_g: float = 0.5,
    collapse_mirrors: bool = True,
    print_output: bool = True,
) -> str:
    if table.empty:
        text = "No notable ablation effects found."
        if print_output:
            print(text)
        return text

    df = table.copy()
    df["metric_label"] = df["metric"].map(METRIC_LABELS).fillna(df["metric"])
    df["abs_g"] = df["hedges_g"].abs()
    df["benefit_g"] = np.where(
        df["interpretation"].eq("better"),
        df["abs_g"],
        -df["abs_g"],
    )

    df = df[df["abs_g"] >= min_abs_g].copy()

    if df.empty:
        text = f"No ablation effects with |g| >= {min_abs_g}."
        if print_output:
            print(text)
        return text

    lines = [
        "=== Ablation summary ===",
        "",
        f"Significant ablation effects, |g| >= {min_abs_g}:",
    ]

    specifics = (
        df.groupby(["ablation_column", "ablation_value"], dropna=False)
        .agg(
            net_g=("benefit_g", "mean"),
            mean_abs_g=("abs_g", "mean"),
        )
        .reset_index()
        .sort_values("net_g", ascending=False)
    )

    for _, r in specifics.iterrows():
        gdf = df[
            df["ablation_column"].eq(r["ablation_column"])
            & df["ablation_value"].eq(r["ablation_value"])
        ]

        net_g = r["net_g"]
        verdict = (
            "mostly beneficial" if net_g > 0.15
            else "mostly harmful" if net_g < -0.15
            else "mixed"
        )

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']}: "
            f"{verdict}; "
            f"better on {metric_list(gdf, 'better')}; "
            f"worse on {metric_list(gdf, 'worse')}; "
            f"net g={net_g:.2f}, mean |g|={r['mean_abs_g']:.2f}"
        )

    if collapse_mirrors:
        kept = []

        for _, gdf in df.groupby(["ablation_column", "metric"], dropna=False):
            better = gdf[gdf["interpretation"].eq("better")]
            pick_from = better if not better.empty else gdf
            kept.append(pick_from.sort_values("abs_g", ascending=False).iloc[0])

        df = pd.DataFrame(kept)

    df = df.sort_values("abs_g", ascending=False).head(top_n)

    lines += ["", "Strongest individual effects:"]

    for _, r in df.iterrows():
        metric = r["metric"]
        verb = "improves" if r["interpretation"] == "better" else "hurts"
        pct = "nan%" if pd.isna(r["pct_delta_mean"]) else f"{r['pct_delta_mean']:+.1f}%"
        direction = "lower is better" if metric in LOWER_IS_BETTER else "higher is better"

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']} {verb} "
            f"{r['metric_label']}: "
            f"{fmt_value(metric, r['group_mean'])} vs {fmt_value(metric, r['other_mean'])} "
            f"({pct}, {effect_strength(r['abs_g'])}, g={r['hedges_g']:.2f}; "
            f"{direction})"
        )

    text = "\n".join(lines)

    if print_output:
        print(text)

    return text


report = analyze_ablation_effects_hedges_g(
    df,
    ablation_columns=ablation_columns,
    metric_columns=metric_columns,
    min_abs_effect=0.2,
)

notable = report[report["abs_effect"] >= 0.5].copy()

text = ablation_effects_to_text_hedges_g(
    notable,
    top_n=40,
    min_abs_g=0.5,
    collapse_mirrors=True,
)

=== Ablation summary ===

Significant ablation effects, |g| >= 0.5:
- isotropy=anisotropic: mostly beneficial; better on LPIPS (+2.55), SSIM (+1.63), PSNR (+1.42); worse on none; net g=1.87, mean |g|=1.87
- dropout=no_dropout: mostly beneficial; better on training time (+2.01), render FPS (+0.69); worse on none; net g=1.35, mean |g|=1.35
- pruning=interleaved_prune_densify: mostly beneficial; better on eval VRAM (+0.75); worse on none; net g=0.75, mean |g|=0.75
- appearance=rgb: mostly beneficial; better on checkpoint size (+2.78), eval VRAM (+0.64); worse on PSNR (-1.15), SSIM (-0.83), LPIPS (-0.52); net g=0.18, mean |g|=1.19
- appearance=sh3: mostly harmful; better on PSNR (+1.15), SSIM (+0.83), LPIPS (+0.52); worse on checkpoint size (-2.78), eval VRAM (-0.64); net g=-0.18, mean |g|=1.19
- pruning=final_pruning: mostly harmful; better on none; worse on eval VRAM (-0.57); net g=-0.57, mean |g|=0.57
- dropout=dropout: mostly harmful; better on none; worse on training time (-2.01), ren

In [46]:
import itertools
import pandas as pd


ABLATION_COLUMNS = [
    "matrix_preset",
    "isotropy",
    "use_usplat",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
]

# Check completeness for the logical best-checkpoint selection.
COMPLETENESS_GROUP_COLUMNS = [
    "dataset_key",
    "scene_name",
]


def check_ablation_grid_complete(
    df: pd.DataFrame,
    ablation_columns: list[str],
    *,
    group_columns: list[str] | None = None,
    print_rows: int | None = 200,
) -> pd.DataFrame:
    group_columns = group_columns or []

    required = group_columns + ablation_columns
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in df: {missing_cols}")

    # Observed valid levels for each ablation dimension.
    levels = {
        c: sorted(df[c].dropna().unique().tolist(), key=str)
        for c in ablation_columns
    }

    for c, vals in levels.items():
        print(f"{c}: {vals}")

    # All expected ablation combinations.
    ablation_grid = pd.DataFrame(
        itertools.product(*(levels[c] for c in ablation_columns)),
        columns=ablation_columns,
    )

    if group_columns:
        group_grid = df[group_columns].drop_duplicates().reset_index(drop=True)

        expected = (
            group_grid.assign(_key=1)
            .merge(ablation_grid.assign(_key=1), on="_key")
            .drop(columns="_key")
        )
    else:
        expected = ablation_grid

    observed = df[required].drop_duplicates()

    missing = expected.merge(
        observed,
        on=required,
        how="left",
        indicator=True,
    )

    missing = (
        missing[missing["_merge"].eq("left_only")]
        .drop(columns="_merge")
        .sort_values(required)
        .reset_index(drop=True)
    )

    duplicate_cases = (
        df.groupby(required, dropna=False)
        .size()
        .reset_index(name="count")
        .query("count > 1")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    expected_n = len(expected)
    observed_n = len(observed)
    missing_n = len(missing)

    print()
    print(f"Expected unique cases: {expected_n}")
    print(f"Observed unique cases: {observed_n}")
    print(f"Missing cases: {missing_n}")
    print(f"Duplicate cases: {len(duplicate_cases)}")

    if missing_n:
        print()
        print("Missing ablation cases:")
        if print_rows is None:
            print(missing.to_string(index=False))
        else:
            print(missing.head(print_rows).to_string(index=False))
            if missing_n > print_rows:
                print(f"... {missing_n - print_rows} more missing cases not shown")

    if len(duplicate_cases):
        print()
        print("Duplicate cases:")
        print(duplicate_cases.head(50).to_string(index=False))

    return missing


missing_cases = check_ablation_grid_complete(
    df_clean,
    ABLATION_COLUMNS,
    group_columns=COMPLETENESS_GROUP_COLUMNS,
)

matrix_preset: ['cartesian']
isotropy: ['anisotropic', 'isotropic']
use_usplat: [False]
appearance: ['rgb', 'sh3']
sorting: ['sort']
pruning: ['early_init_pruning', 'final_pruning', 'interleaved_prune_densify', 'no_pruning']
dropout: ['dropout', 'no_dropout']
ess: ['ess', 'no_ess']

Expected unique cases: 128
Observed unique cases: 128
Missing cases: 0
Duplicate cases: 0
